In [1]:
import calendar
import time
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')

In [2]:
# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

In [40]:
# 설정값들
ticker = 'AAPL'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

data_start_date = '2005-01-01'

In [41]:
print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_raw = fetch_db_revenue_data(ticker, db_info)
db_revenue_df = db_revenue_raw.loc[db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()]

# db_revenue_df = db_revenue_raw.drop_duplicates(subset=['date_month_end', 'revenue_billions'], keep='first')
mereged_rev_data = pd.merge(fmp_revenue_df, db_revenue_df, on = ['ticker', 'date_month_end'], how='outer')

rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= data_start_date]
rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])

전처리 과정 테스트 시작
대상 종목: AAPL

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 160건


In [42]:
rev_data = mereged_rev_data[mereged_rev_data['date_month_end'] >= data_start_date]
rev_data['revenue_billions_x'] = rev_data['revenue_billions_x'].fillna(rev_data['revenue_billions_y'])
rev_data.tail(24)

,ticker,date_x,calendar_year,period,revenue,revenue_billions_x,date_month_end,date_y,revenue_billions_y
137,AAPL,2019-09-28,2019,Q4,6.404000e+10,64.04,2019-09-30,2019-09-30,64.040
138,AAPL,2019-12-28,2020,Q1,9.181900e+10,91.82,2019-12-31,2019-12-31,91.819
139,AAPL,2020-03-28,2020,Q2,5.831300e+10,58.31,2020-03-31,2020-03-31,58.313
140,AAPL,2020-06-27,2020,Q3,5.968500e+10,59.69,2020-06-30,2020-06-30,59.685
141,AAPL,2020-09-26,2020,Q4,6.469800e+10,64.70,2020-09-30,2020-09-30,64.698
142,AAPL,2020-12-26,2021,Q1,1.114390e+11,111.44,2020-12-31,2020-12-31,111.439
143,AAPL,2021-03-27,2021,Q2,8.958400e+10,89.58,2021-03-31,2021-03-31,89.584
144,AAPL,2021-06-26,2021,Q3,8.143400e+10,81.43,2021-06-30,2021-06-30,81.434
145,AAPL,2021-09-25,2021,Q4,8.336000e+10,83.36,2021-09-30,2021-09-30,83.360
146,AAPL,2021-12-25,2022,Q1,1.239450e+11,123.94,2021-12-31,2021-12-31,123.945


In [43]:
rev_data

,ticker,date_x,calendar_year,period,revenue,revenue_billions_x,date_month_end,date_y,revenue_billions_y
79,AAPL,2005-03-26,2005,Q2,3.243000e+09,3.24,2005-03-31,2005-03-31,3.243
80,AAPL,2005-06-25,2005,Q3,3.520000e+09,3.52,2005-06-30,2005-06-30,3.520
81,AAPL,2005-09-24,2005,Q4,3.678000e+09,3.68,2005-09-30,2005-09-30,3.678
82,AAPL,2005-12-31,2006,Q1,5.749000e+09,5.75,2005-12-31,2005-12-31,5.749
83,AAPL,2006-04-01,2006,Q2,4.359000e+09,4.36,2006-03-31,2006-03-31,4.359
...,...,...,...,...,...,...,...,...,...
156,AAPL,2024-06-29,2024,Q3,8.577700e+10,85.78,2024-06-30,2024-06-30,85.777
157,AAPL,2024-09-28,2024,Q4,9.493000e+10,94.93,2024-09-30,2024-09-30,94.930
158,AAPL,2024-12-28,2025,Q1,1.243000e+11,124.30,2024-12-31,2024-12-31,124.300
159,AAPL,2025-03-29,2025,Q2,9.535900e+10,95.36,2025-03-31,2025-03-31,95.359


In [9]:
fmp_revenue_df

,ticker,date,calendar_year,period,revenue,revenue_billions,date_month_end
0,AMAT,1985-10-31,1985,Q4,35300000,0.04,1985-10-31
1,AMAT,1986-01-31,1986,Q1,33900000,0.03,1986-01-31
2,AMAT,1986-04-30,1986,Q2,36000000,0.04,1986-04-30
3,AMAT,1986-07-31,1986,Q3,39200000,0.04,1986-07-31
4,AMAT,1986-10-31,1986,Q4,40300000,0.04,1986-10-31
...,...,...,...,...,...,...,...
154,AMAT,2024-07-28,2024,Q3,6778000000,6.78,2024-07-31
155,AMAT,2024-10-27,2024,Q4,7045000000,7.04,2024-10-31
156,AMAT,2025-01-26,2025,Q1,7166000000,7.17,2025-01-31
157,AMAT,2025-04-27,2025,Q2,7100000000,7.10,2025-04-30


In [13]:
db_revenue_df = db_revenue_df.drop_duplicates(subset=['date'], keep='first')
db_revenue_df

,ticker,date,date_month_end,revenue_billions
0,AMAT,2000-01-31,2000-01-31,1.722190
1,AMAT,2000-02-29,2000-02-29,1.722190
2,AMAT,2000-03-31,2000-03-31,1.722190
3,AMAT,2000-04-30,2000-04-30,2.190031
4,AMAT,2000-05-31,2000-05-31,2.190031
...,...,...,...,...
549,AMAT,2025-03-31,2025-03-31,7.166000
550,AMAT,2025-04-30,2025-04-30,7.100000
551,AMAT,2025-05-31,2025-05-31,7.100000
552,AMAT,2025-06-30,2025-06-30,7.100000
